In [1]:
%load_ext autoreload
%autoreload 2

## Path/Import Cell

In [2]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

## UI Function Cell

In [3]:
from intuition_engine.examples import EXAMPLES
from intuition_engine.pipeline import analyze_equation

def launch_engine_ui():

    import ipywidgets as widgets
    import sympy as sp
    from IPython.display import display, Markdown, clear_output, Math

    # Single input box for the equation; seed with a representative example
    custom_input = widgets.Textarea(
        value=EXAMPLES["Driven damped oscillator"],
        description="Equation:",
        layout=widgets.Layout(width="900px", height="120px")
    )

    analyze_button = widgets.Button(
        description="Analyze Equation",
        button_style="success"
    )

    output_area = widgets.Output()

    def safe_latex(expr):
        if expr is None:
            return r"\text{None}"
        try:
            return sp.latex(expr)
        except Exception:
            return str(expr)

    def _render_system_report(report):
        ps = report.parsed_system
        si = report.system_info
        t = ps.independent_variable
        display(Markdown("## System of differential equations"))
        for func, rhs in ps.equations:
            display(Math(rf"{safe_latex(sp.diff(func(t), t))} = {safe_latex(rhs)}"))
        display(Markdown("## Solution (general)"))
        if getattr(report, 'solution', None):
            sols = report.solution if isinstance(report.solution, list) else [report.solution]
            for eq in sols:
                try:
                    display(Math(safe_latex(eq)))
                except Exception:
                    display(Markdown(f"- {eq}"))
        else:
            err = getattr(report, "solution_error", None)
            if err:
                display(Markdown("No general solution computed: " + err))
            else:
                display(Markdown("No general solution computed."))
        display(Markdown("## Coefficient matrix A and vector b"))
        display(Markdown("Normal form: dx/dt = A x + b"))
        try:
            display(Math(rf"A = {safe_latex(si.matrix_A)}"))
            display(Math(rf"b = {safe_latex(si.vector_b)}"))
        except Exception:
            display(Markdown("- Matrix A: " + str(si.matrix_A)))
            display(Markdown("- Vector b: " + str(si.vector_b)))
        display(Markdown("## Eigenvalues"))
        for i, ev in enumerate(si.eigenvalues, start=1):
            try:
                display(Math(rf"\\lambda_{i} = {safe_latex(ev)}"))
            except Exception:
                display(Markdown(f"- lambda_{i} = {ev}"))
        display(Markdown("## Trace and determinant"))
        if getattr(si, 'trace', None) is not None:
            try:
                display(Math(rf"\\operatorname{{tr}}(A) = {safe_latex(si.trace)}"))
            except Exception:
                display(Markdown("- tr(A) = " + str(si.trace)))
        if getattr(si, 'determinant', None) is not None:
            try:
                display(Math(rf"\\det(A) = {safe_latex(si.determinant)}"))
            except Exception:
                display(Markdown("- det(A) = " + str(si.determinant)))
        display(Markdown("## Eigenvectors"))
        try:
            evecs = getattr(si, 'eigenvectors', None) or []
            if not evecs:
                display(Markdown("No eigenvectors computed."))
            else:
                for idx, (ev, vects) in enumerate(evecs, start=1):
                    display(Markdown("**Eigenvalue " + str(idx) + ":**"))
                    try:
                        display(Math(safe_latex(ev)))
                    except Exception:
                        display(Markdown(str(ev)))
                    display(Markdown("Eigenvector(s):"))
                    for v in vects:
                        try:
                            display(Math(safe_latex(v)))
                        except Exception:
                            display(Markdown("- " + str(v)))
        except Exception as ex:
            display(Markdown("Eigenvectors error: " + str(ex)))
        display(Markdown("## Stability"))
        display(Markdown(si.stability_summary))
        if si.notes:
            for n in si.notes:
                display(Markdown(f"- {n}"))
        display(Markdown("## Regime insights (from eigenvalues)"))
        for insight in si.regime_insights:
            display(Markdown(f"- {insight}"))
        display(Markdown("## Classification"))
        display(Markdown(f"- Family: `{report.classification.family}`"))
        display(Markdown(f"- Linear, constant coefficients: {report.classification.has_constant_coefficients}"))
        display(Markdown("## What this system could represent physically"))
        for sys in report.physical_systems:
            display(Markdown(f"- {sys}"))

    def render_report(report):
        parsed = report.parsed
        classification = report.classification
        features = report.features
        scaling = report.scaling
        regimes = report.regimes
        validation = report.validation
        physical_systems = report.physical_systems

        if report.system_info is not None:
            _render_system_report(report)
            return

        display(Markdown("## Input / Canonical Form"))
        display(Math(rf"{safe_latex(parsed.lhs)} = {safe_latex(parsed.rhs)}"))
        display(Math(rf"{safe_latex(parsed.canonical_expr)} = 0"))

        display(Markdown("## Solution (general)"))
        if getattr(report, 'solution', None) is not None:
            sols = report.solution if isinstance(report.solution, list) else [report.solution]
            for eq in sols:
                try:
                    display(Math(safe_latex(eq)))
                except Exception:
                    display(Markdown(f"- {eq}"))
        else:
            err = getattr(report, "solution_error", None)
            if err:
                display(Markdown("No general solution computed: " + err))
            else:
                display(Markdown("No general solution computed."))

        display(Markdown("## Validation"))

        level = getattr(validation, "support_level", "unrecognized" if not validation.is_supported else "partial")
        if level == "full":
            display(Markdown("- ✅ **Fully supported** — full interpretation available (second-order linear constant-coeff ODE or linear system)."))
        elif level == "partial":
            display(Markdown("- ⚠️ **Partially supported** — recognized but interpretation may be incomplete."))
        else:
            display(Markdown("- ❌ **Not in a recognized family** — equation is outside current engine families."))
            display(Markdown("**Warnings**"))
            for w in validation.warnings:
                display(Markdown(f"- {w}"))

        display(Markdown("## Classification"))
        display(Markdown(
            f"""
            - **Order:** {classification.order}
            - **Linear:** {classification.is_linear}
            - **Constant coefficients:** {classification.has_constant_coefficients}
            - **Forced:** {classification.is_forced}
            - **Recognized family:** `{classification.family}`
            """
        ))

        if classification.notes:
            display(Markdown("**Notes**"))
            for note in classification.notes:
                display(Markdown(f"- {note}"))

        display(Markdown("## Extracted Features"))
        display(Math(rf"a = {safe_latex(features.a)}"))
        display(Math(rf"b = {safe_latex(features.b)}"))
        display(Math(rf"c = {safe_latex(features.c)}"))
        display(Math(rf"f(t) = {safe_latex(features.forcing)}"))
        display(Math(rf"\Delta = {safe_latex(features.discriminant)}"))
        display(Math(rf"\omega_n = {safe_latex(features.natural_frequency)}"))
        display(Math(rf"\zeta = {safe_latex(features.damping_ratio)}"))
        display(Math(rf"\tau_d \sim {safe_latex(features.damping_timescale)}"))
        display(Math(rf"\text{{forcing frequency}} = {safe_latex(features.forcing_frequency)}"))
        display(Math(rf"\text{{Characteristic polynomial: }} {safe_latex(features.characteristic_polynomial)}"))

        display(Markdown("## Scaling / Dimensionless Structure"))

        display(Math(rf"\omega_n = {safe_latex(scaling.natural_frequency)}"))
        display(Math(rf"\zeta = {safe_latex(scaling.damping_ratio)}"))
        
        if scaling.forcing_ratio is not None:
            display(Math(rf"r = \frac{{\omega}}{{\omega_n}} = {safe_latex(scaling.forcing_ratio)}"))
        
        if scaling.notes:
            for note in scaling.notes:
                display(Markdown(f"- {note}"))

        if features.roots is not None:
            display(Markdown("**Characteristic roots**"))
            for i, root in enumerate(features.roots, start=1):
                display(Math(rf"r_{i} = {safe_latex(root)}"))

        display(Markdown("## Regime Insights"))
        for r in regimes:
            display(Markdown(f"**{r.label}**"))
            try:
                display(Math(rf"{safe_latex(r.condition)}"))
            except Exception:
                display(Markdown(f"- Condition: `{r.condition}`"))
            display(Markdown(f"- Meaning: {r.meaning}"))

        display(Markdown("## What this equation could represent physically"))
        for sys in physical_systems:
            display(Markdown(f"- {sys}"))
            
    def on_click(_):
        with output_area:
            clear_output()
            try:
                eq = custom_input.value.strip()
                if not eq:
                    display(Markdown("## Error"))
                    display(Markdown("Please enter a differential equation in the text box."))
                    return
                report = analyze_equation(eq)
                render_report(report)
            except Exception as e:
                display(Markdown("## Error"))
                print(type(e).__name__ + ":", e)

    analyze_button.on_click(on_click)

    display(
        widgets.VBox([
            widgets.HTML("<h1>Physics Intuition Engine</h1>"),
            widgets.HTML("<p>Paste a single ODE or a system of ODEs (separate equations by <code>;</code> or newline), then click Analyze.</p>"),
            custom_input,
            analyze_button,
            output_area
        ])
    )

## `launch_engine_ui()` Cell

In [4]:
launch_engine_ui()